## 01_config


In [ ]:
# Cell 1 — Install dependencies FIRST before anything else
!pip install pysodmetrics -q
!pip install gdown -q
!pip install -U huggingface_hub -q
import py_sod_metrics
print(f"Successfully imported py_sod_metrics")
test_mae = py_sod_metrics.MAE()
print(f"MAE Metric object successfully created!")

import os, json, shutil, hashlib

# Configuration & Modes
# RUN_MODE strictly governs the allowed execution path.
# Allowed: "VALIDATE", "TRAIN", "RESUME", "EVALUATE"
RUN_MODE = "EVALUATE"

# --- Training data (unchanged — still Google Drive, out of scope of the HF migration) ---
DATA_SOURCE = "GOOGLE_DRIVE"
DATA_FILE_ID = "1SSELvRYI-cwd9mzA8dWLbv4o1IffjkoW"

# --- Hugging Face Hub is now the single source of truth for code + checkpoints ---
# HF_TOKEN is read from a Kaggle Secret (Add-ons > Secrets > add "HF_TOKEN"),
# never hardcoded. HF_REPO_ID is the one repo holding both code/ and checkpoints/.
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO_ID = "Avi2006/spatial-moe-results"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_REPO_ID"] = HF_REPO_ID

PROJECT_ROOT = "/kaggle/working/spatial_moe_sod"
CHECKPOINT_ROOT = "/kaggle/working/WXSOD_Checkpoints"
PREFLIGHT_ROOT = "/kaggle/working/WXSOD_Preflight"
RESUME_TEST_ROOT = "/kaggle/working/WXSOD_ResumeTest"
NUM_GPUS = 2
FINAL_EPOCHS = 50

# These get exported so train_ddp.py (run via torchrun as a subprocess)
# picks up the exact same checkpoint location and HF repo.
os.environ["CHECKPOINT_ROOT"] = CHECKPOINT_ROOT
os.environ["PREFLIGHT_ROOT"] = PREFLIGHT_ROOT

# Global state for dynamic final audit gate
GATES = {
    "ENV_CHECK": "NOT_RUN",
    "GPU_CHECK": "NOT_RUN",
    "DATA_CHECK": "NOT_RUN",
    "PROJECT_CHECK": "NOT_RUN",
    "DEPENDENCY_CHECK": "NOT_RUN",
    "PVT_CHECK": "NOT_RUN",
    "STATIC_CHECK": "NOT_RUN",
    "TRANSFORM_CHECK": "NOT_RUN",
    "MODEL_CHECK": "NOT_RUN",
    "MOE_CHECK": "NOT_RUN",
    "LOSS_CHECK": "NOT_RUN",
    "OPT_CHECK": "NOT_RUN",
    "DDP_CHECK": "NOT_RUN",
    "MEMORY_CHECK": "NOT_RUN",
    "CHECKPOINT_CHECK": "NOT_RUN",
    "RESUME_CHECK": "NOT_RUN",
    "PREFLIGHT_DRY_RUN_CHECK": "NOT_RUN",
    "TORCHRUN_1GPU_CHECK": "NOT_RUN",
    "RESUME_PREFLIGHT_CHECK": "NOT_RUN"
}

def mark_gate(gate, status, msg="", evidence="", run_id=None, config_hash=None):
    if GATES.get(gate) == "FAIL" and status == "PASS":
        print(f"[{gate}] FAIL -> PASS transition authorized (run_id: {run_id}, config_hash: {config_hash})")
    print(f"[{gate}] -> {status} {msg}")
    GATES[gate] = status


# --- Auto-fetch checkpoint from Hugging Face Hub (replaces the old Google Drive flow) ---
# Standalone on purpose: the project's own src/hf_sync.py isn't deployed yet
# at this point in the notebook (that happens in 05_project_deploy), so this
# duplicates its checkpoint-pull logic in miniature rather than importing it.
def _sha256_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def fetch_checkpoint_from_hf(repo_id, token, dest_root):
    from huggingface_hub import hf_hub_download
    from huggingface_hub.utils import EntryNotFoundError
    os.makedirs(dest_root, exist_ok=True)

    try:
        manifest_path = hf_hub_download(
            repo_id=repo_id, repo_type="dataset",
            filename="checkpoints/checkpoint_manifest.json", token=token,
        )
    except EntryNotFoundError:
        mark_gate("CHECKPOINT_CHECK", "NOT_RUN", msg="No checkpoint manifest on HF yet — training from scratch.")
        return False

    with open(manifest_path) as f:
        manifest = json.load(f)
    files_meta = manifest.get("files", {})

    fetched_any = False
    for name in ("latest.pth", "best.pth"):
        meta = files_meta.get(name)
        if meta is None:
            continue
        local_path = os.path.join(dest_root, name)
        if os.path.exists(local_path) and _sha256_file(local_path) == meta.get("sha256"):
            print(f"{name}: local copy already matches HF (sha256 match), skipping download.")
            fetched_any = True
            continue
        try:
            downloaded = hf_hub_download(
                repo_id=repo_id, repo_type="dataset",
                filename=f"checkpoints/{name}", token=token,
            )
        except EntryNotFoundError:
            continue
        shutil.copy2(downloaded, local_path)
        actual = _sha256_file(local_path)
        if meta.get("sha256") and actual != meta["sha256"]:
            mark_gate("CHECKPOINT_CHECK", "FAIL", msg=f"{name}: sha256 mismatch after download")
            raise RuntimeError(f"{name}: downloaded sha256 {actual} != manifest sha256 {meta['sha256']}")
        print(f"{name}: pulled from HF -> {local_path}")
        fetched_any = True

    if fetched_any:
        shutil.copy2(manifest_path, os.path.join(dest_root, "checkpoint_manifest.json"))
        mark_gate("CHECKPOINT_CHECK", "PASS", msg=f"Fetched checkpoint(s) from {repo_id} -> {dest_root}")
    else:
        mark_gate("CHECKPOINT_CHECK", "NOT_RUN", msg="Manifest present but no checkpoint files found.")
    return fetched_any


CHECKPOINT_AVAILABLE = fetch_checkpoint_from_hf(HF_REPO_ID, HF_TOKEN, CHECKPOINT_ROOT)

if CHECKPOINT_AVAILABLE:
    if RUN_MODE == "TRAIN":
        print("Checkpoint found on HF — switching RUN_MODE to RESUME")
        RUN_MODE = "RESUME"
else:
    print("No checkpoint found on HF — will train from scratch.")


## 02_environment


In [7]:
import torch
import os
import shutil

print("1. Environment & GPU Info")
print("PyTorch Version:", torch.__version__)
print("CUDA Version:", torch.version.cuda)
num_gpus = torch.cuda.device_count()
print("GPU Count:", num_gpus)
for i in range(num_gpus):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Disk space check
total, used, free = shutil.disk_usage("/kaggle/working")
free_gb = free / (1024**3)
print(f"Available Disk Space: {free_gb:.2f} GB")
if free_gb < 5.0:
    mark_gate("ENV_CHECK", "FAIL", "Insufficient disk space.")
    raise RuntimeError("At least 5 GB required in /kaggle/working.")

if num_gpus < NUM_GPUS and RUN_MODE in ["VALIDATE", "TRAIN", "RESUME"]:
    mark_gate("GPU_CHECK", "FAIL", f"Expected {NUM_GPUS} GPUs")
    raise RuntimeError(f"Expected {NUM_GPUS} GPUs")
else:
    mark_gate("GPU_CHECK", "PASS")
    mark_gate("ENV_CHECK", "PASS")



1. Environment & GPU Info
PyTorch Version: 2.10.0+cu128
CUDA Version: 12.8
GPU Count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Available Disk Space: 14.01 GB
[GPU_CHECK] -> PASS 
[ENV_CHECK] -> PASS 


## 03_dataset_acquire


In [8]:
import subprocess

def is_dataset_valid(path):
    def check_root(r):
        reqs = ["train_sys/input", "train_sys/gt", "test_sys/input", "test_sys/gt", "test_real/input", "test_real/gt"]
        return all(os.path.isdir(os.path.join(r, req)) for req in reqs)
        
    if check_root(path):
        return path
    for root, dirs, _ in os.walk(path):
        if check_root(root):
            return root
    return None

EXTRACT_PATH = "/kaggle/working/WXSDO_data"

valid_root = is_dataset_valid(EXTRACT_PATH)
if valid_root is None:
    print("Dataset not found locally, downloading...")
    ZIP_PATH = "/kaggle/working/dataset.zip"
    if not os.path.exists(ZIP_PATH):
        subprocess.run(["pip", "install", "-q", "gdown"], check=True)
        subprocess.run(["gdown", DATA_FILE_ID, "-O", ZIP_PATH], check=True)
    os.makedirs(EXTRACT_PATH, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", EXTRACT_PATH], check=True)
    valid_root = is_dataset_valid(EXTRACT_PATH)

if valid_root is None:
    mark_gate("DATA_CHECK", "FAIL")
    raise RuntimeError("Dataset extraction completed but valid WXSOD root was not found.")
else:
    print("Dataset successfully extracted and validated at:", valid_root)



Dataset successfully extracted and validated at: /kaggle/working/WXSDO_data/WXSDO_data


In [9]:
# import torch

# # Path to your best checkpoint
# checkpoint_path = "/kaggle/working/WXSOD_Checkpoints/best.pth"

# try:
#     # Added weights_only=False here
#     checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    
#     print(f"✅ Current Best Model is from Epoch: {checkpoint['epoch'] + 1}")
#     print(f"🏆 Current Best MAE: {checkpoint['best_metric']:.4f}")
# except Exception as e:
#     print(f"Error loading checkpoint: {e}")


✅ Current Best Model is from Epoch: 15
🏆 Current Best MAE: 0.0196


## 04_dataset_validate


In [10]:
import glob

def check_stems(split_path):
    input_files = glob.glob(os.path.join(split_path, "input", "*.*"))
    gt_files = glob.glob(os.path.join(split_path, "gt", "*.*"))
    
    valid_exts = {".png", ".jpg", ".jpeg"}
    input_files = [f for f in input_files if os.path.splitext(f)[1].lower() in valid_exts]
    gt_files = [f for f in gt_files if os.path.splitext(f)[1].lower() in valid_exts]
    
    input_stems = {os.path.splitext(os.path.basename(f))[0] for f in input_files}
    gt_stems = {os.path.splitext(os.path.basename(f))[0] for f in gt_files}
    
    missing_gt = input_stems - gt_stems
    missing_input = gt_stems - input_stems
    
    if missing_gt or missing_input:
        return False, len(missing_gt), len(missing_input)
    return True, len(input_stems), len(gt_stems)

splits = {"train_sys": 12891, "test_sys": 1500, "test_real": 554}
all_valid = True
for split, req_count in splits.items():
    split_path = os.path.join(valid_root, split)
    if not os.path.exists(split_path):
        print(f"{split} missing!")
        all_valid = False
        continue
    valid, m_gt, m_in = check_stems(split_path)
    if not valid:
        print(f"{split} stem mismatch: {m_gt} missing GT, {m_in} missing Inputs")
        all_valid = False
    else:
        print(f"{split}: {m_in} pairs verified perfectly.")
        if RUN_MODE in ["VALIDATE", "TRAIN", "EVALUATE"] and m_in != req_count:
            print(f"{split} count mismatch. Expected {req_count}, got {m_in}.")
            all_valid = False

if all_valid:
    mark_gate("DATA_CHECK", "PASS")
else:
    mark_gate("DATA_CHECK", "FAIL")
    raise RuntimeError("Dataset integrity check failed.")



train_sys: 12891 pairs verified perfectly.
test_sys: 1500 pairs verified perfectly.
test_real: 554 pairs verified perfectly.
[DATA_CHECK] -> PASS 


## 05_project_deploy


In [ ]:
import sys
import json
import hashlib
import shutil
import os
import zipfile

os.makedirs(PROJECT_ROOT, exist_ok=True)

def compute_dir_hash(root_dir, include=("src", "tests", "experiments", "requirements.txt", "pyproject.toml", "train.py")):
    """Deterministic SHA256 over (relative_path, file_bytes) for the real
    project files. Must match src/hf_sync.compute_dir_hash exactly — this is
    intentionally duplicated here (src/ isn't deployed onto disk yet at this
    point), and both copies are the single check that catches staleness,
    corruption, or a partial HF upload."""
    hasher = hashlib.sha256()
    file_list = []
    for item in include:
        full = os.path.join(root_dir, item)
        if os.path.isdir(full):
            for base, _, files in os.walk(full):
                for fn in files:
                    fp = os.path.join(base, fn)
                    rel = os.path.relpath(fp, root_dir)
                    file_list.append((rel, fp))
        elif os.path.isfile(full):
            file_list.append((item, full))
    file_list.sort(key=lambda x: x[0])
    for rel, fp in file_list:
        hasher.update(rel.replace(os.sep, "/").encode("utf-8"))
        with open(fp, "rb") as f:
            hasher.update(f.read())
    return hasher.hexdigest()

def _sha256_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

# --- Pull code from Hugging Face Hub (replaces the Kaggle-Dataset zip/manifest input) ---
from huggingface_hub import hf_hub_download

HF_CODE_CACHE = "/kaggle/working/hf_code_cache"
os.makedirs(HF_CODE_CACHE, exist_ok=True)

manifest_path = hf_hub_download(
    repo_id=HF_REPO_ID, repo_type="dataset",
    filename="code/project_manifest.json", token=HF_TOKEN, local_dir=HF_CODE_CACHE,
)
with open(manifest_path) as f:
    manifest = json.load(f)

zip_path = hf_hub_download(
    repo_id=HF_REPO_ID, repo_type="dataset",
    filename="code/spatial_moe_sod_code.zip", token=HF_TOKEN, local_dir=HF_CODE_CACHE,
)
actual_archive_sha = _sha256_file(zip_path)
if manifest.get("archive_sha256") and actual_archive_sha != manifest["archive_sha256"]:
    mark_gate("PROJECT_CHECK", "FAIL", "Downloaded zip sha256 does not match manifest.")
    raise RuntimeError(
        f"Downloaded zip sha256 mismatch! expected={manifest['archive_sha256']} actual={actual_archive_sha}. "
        "Re-run `python -m src.hf_sync push-code` from VS Code and try again."
    )

if os.path.exists(PROJECT_ROOT):
    shutil.rmtree(PROJECT_ROOT)
os.makedirs(PROJECT_ROOT, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(PROJECT_ROOT)
print(f"Deployed from Hugging Face ({HF_REPO_ID}) -> {PROJECT_ROOT}")

# Post-unzip existence assertions
required_files = ["src/train_ddp.py", "src/smoke_test.py", "src/model.py", "src/optimization.py", "src/evaluate.py", "src/hf_sync.py"]
for f in required_files:
    if not os.path.exists(os.path.join(PROJECT_ROOT, f)):
        mark_gate("PROJECT_CHECK", "FAIL")
        raise RuntimeError(f"Payload corrupted: missing {f}")

# CONTENT VERIFICATION -- this is the check that actually matters. It hashes
# the files that were actually deployed, so it catches staleness, corruption,
# or a partial upload regardless of how the bytes got here.
expected_hash = manifest.get("source_content_sha256")
if expected_hash:
    actual_hash = compute_dir_hash(PROJECT_ROOT)
    print("Expected source_content_sha256:", expected_hash)
    print("Actual   source_content_sha256:", actual_hash)
    if actual_hash != expected_hash:
        mark_gate("PROJECT_CHECK", "FAIL", "Deployed source content does not match manifest!")
        raise RuntimeError(
            "Deployed code does not match project_manifest.json's source_content_sha256. "
            "The code actually running is NOT the code you think you pushed. "
            "Run `python -m src.hf_sync push-code` from VS Code again and re-run this cell."
        )
else:
    mark_gate("PROJECT_CHECK", "FAIL", "Manifest missing source_content_sha256.")
    raise RuntimeError(
        "project_manifest.json has no source_content_sha256 field — this looks like a manifest "
        "from before the HF migration. Push code again with the current src/hf_sync.py."
    )

print("Project source code deployed to:", PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# Purge any 'src.*' modules already cached in THIS kernel session, so re-running
# only the deploy cell (instead of a full kernel restart) picks up fresh code.
for _mod_name in list(sys.modules.keys()):
    if _mod_name == "src" or _mod_name.startswith("src."):
        del sys.modules[_mod_name]

mark_gate("PROJECT_CHECK", "PASS")


## 06_dependencies


In [12]:
import importlib
deps = ["torch", "torchvision", "timm", "albumentations", "cv2", "numpy"]
env_data = {}
all_passed = True
for dep in deps:
    try:
        mod = importlib.import_module(dep)
        env_data[dep] = getattr(mod, "__version__", "unknown")
    except ImportError:
        print(f"Missing dependency: {dep}. Installing...")
        subprocess.run(["pip", "install", "-q", dep], check=True)
        try:
            mod = importlib.import_module(dep)
            env_data[dep] = getattr(mod, "__version__", "unknown")
        except ImportError:
            all_passed = False

if all_passed:
    with open(os.path.join(PROJECT_ROOT, "environment.json"), "w") as f:
        json.dump(env_data, f)
    mark_gate("DEPENDENCY_CHECK", "PASS")
else:
    mark_gate("DEPENDENCY_CHECK", "FAIL")
    raise RuntimeError("Failed to resolve dependencies.")

[DEPENDENCY_CHECK] -> PASS 


## 07_pretrained_b4


In [13]:
if RUN_MODE in ["VALIDATE", "TRAIN"]:
    from src.backbone import MultiScaleBackbone
    try:
        model1 = MultiScaleBackbone(model_name='pvt_v2_b4', pretrained=True, d=256)
        model2 = MultiScaleBackbone(model_name='pvt_v2_b4', pretrained=True, d=256)
        
        dummy_input = torch.randn(1, 3, 384, 384)
        feats = model1(dummy_input)
        
        assert torch.isfinite(feats['res_4']).all()
        assert torch.isfinite(feats['res_8']).all()
        assert torch.isfinite(feats['res_16']).all()
        assert feats['res_4'].shape == (1, 256, 96, 96), "Y4 shape incorrect"
        assert feats['res_8'].shape == (1, 256, 48, 48), "Y8 shape incorrect"
        assert feats['res_16'].shape == (1, 256, 24, 24), "Y16 shape incorrect"
        
        del model1, model2
        mark_gate("PVT_CHECK", "PASS")
    except Exception as e:
        mark_gate("PVT_CHECK", "FAIL")
        raise RuntimeError(f"PVT Instantiation failed: {e}")



## 08_static_checks


In [14]:
if RUN_MODE in ["VALIDATE", "TRAIN"]:
    try:
        import src.model
        import src.train_ddp
        import src.loss
        mark_gate("STATIC_CHECK", "PASS")
    except ImportError as e:
        mark_gate("STATIC_CHECK", "FAIL")
        raise e



## 09_data_transform


In [15]:
if RUN_MODE in ["VALIDATE", "TRAIN"]:
    from src.dataset import WXSODDataset
    try:
        ds = WXSODDataset(valid_root, split="train")
        
        # Test synthetic geometric mask alignment
        import numpy as np
        
        test_shapes = [(800, 200), (200, 800), (400, 400), (400, 287), (287, 400), (320, 350)]
        for h, w in test_shapes:
            syn_img = np.zeros((h, w, 3), dtype=np.uint8)
            syn_mask = np.zeros((h, w), dtype=np.uint8)
            if w == 800: syn_mask[:, :400] = 255
            
            # Use the dataset's internal methods to mimic __getitem__
            img_pad, mask_pad, _ = ds._aspect_preserving_resize_pad(syn_img, syn_mask)
            edge_pad = ds._compute_edge_map(mask_pad)
            
            assert img_pad.shape == (384, 384, 3)
            assert mask_pad.shape == (384, 384)
            assert edge_pad.shape == (384, 384)
            
            if w == 800 and h == 200:
                assert (mask_pad[:, 192:] == 0).all(), "Mask alignment failed during resize/pad"
        
        for idx in [0, len(ds)//2, len(ds)-1]:
            samp = ds[idx]
            assert samp['image'].shape == (3, 384, 384)
            assert samp['mask'].shape == (1, 384, 384)
            
        # Check split identity for leakage guard
        assert ds.split == "train", "Leakage guard: Expected train split identity."
        mark_gate("TRANSFORM_CHECK", "PASS")
    except Exception as e:
        mark_gate("TRANSFORM_CHECK", "FAIL")
        raise e



## 10_model_shapes


In [16]:
if RUN_MODE in ["VALIDATE", "TRAIN"]:
    from src.model import SpatialMoESODNet
    try:
        model = SpatialMoESODNet(dim=256)
        dummy_input = torch.randn(1, 3, 384, 384)
        out, moe_outputs = model(dummy_input)
        
        assert torch.isfinite(out.saliency_logits).all(), "saliency_logits not finite"
        assert torch.isfinite(out.boundary_logits).all(), "boundary_logits not finite"
        for m_out in moe_outputs:
            assert torch.isfinite(m_out.features).all(), "routed features not finite"
            assert torch.isfinite(m_out.routing_probs).all(), "router probabilities not finite"
            assert torch.isfinite(m_out.entropy).all(), "entropy not finite"
            
        # The true model architecture expects Y4, Y8, Y16 inside backbone forward
        feats = model.backbone(dummy_input)
        assert feats['res_4'].shape == (1, 256, 96, 96), "Y4 shape incorrect"
        assert feats['res_8'].shape == (1, 256, 48, 48), "Y8 shape incorrect"
        assert feats['res_16'].shape == (1, 256, 24, 24), "Y16 shape incorrect"
        mark_gate("MODEL_CHECK", "PASS")
    except Exception as e:
        mark_gate("MODEL_CHECK", "FAIL")
        raise e



## 11_moe_loss


In [17]:
# Evaluated internally via smoke tests. Setting pseudo-gate to be fulfilled later.
print("Deferring loss assertions to smoke testing.")



Deferring loss assertions to smoke testing.


## 12_optimization


In [18]:
# Evaluated internally via smoke tests.
print("Deferring optimization assertions to smoke testing.")



Deferring optimization assertions to smoke testing.


## 14_smoke_2gpu


In [19]:
if RUN_MODE in ["VALIDATE", "TRAIN"]:
    with open(os.path.join(PROJECT_ROOT, "experiments/baseline_v1.json"), "r") as f:
        canonical_cfg = json.load(f)
    for _m in list(sys.modules.keys()):
        if _m == "src" or _m.startswith("src."):
            del sys.modules[_m]
    from src.train_ddp import get_config_hash
    CURRENT_CONFIG_HASH = get_config_hash(canonical_cfg, "model_config_hash")
    CURRENT_RUN_ID = canonical_cfg.get("run_id") or "test_run_id"

    print("Running 2-GPU DDP Smoke Test...")
    res = subprocess.run(["torchrun", "--nproc_per_node=2", "-m", "src.smoke_test", "--mode", "ddp", "--data_root", valid_root, "--result_file", "test_2gpu.json", "--run_id", CURRENT_RUN_ID, "--config_hash", CURRENT_CONFIG_HASH], cwd=PROJECT_ROOT)
    with open(os.path.join(PROJECT_ROOT, "test_2gpu.json"), "r") as f:
        data = json.load(f)
        if data.get("status") == "PASS" and data.get("world_size") == 2 and set(data.get("ranks_completed", [])) == {0, 1}:
            mark_gate("DDP_CHECK", "PASS")
            mark_gate("MOE_CHECK", data.get("moe_check", "FAIL"), "Sparsity/Token Identity")
            mark_gate("LOSS_CHECK", data.get("loss_check", "FAIL"), "Loss Gradients/Behavior")
            mark_gate("OPT_CHECK", data.get("optimizer_check", "FAIL"), "Optimizer Groups/Behavior")
        else:
            mark_gate("DDP_CHECK", "FAIL")
            raise RuntimeError(f"DDP Smoke Failed: {data}")



## 15_production_calibration


In [20]:
if RUN_MODE in ["VALIDATE", "TRAIN"]:
    print("Running 2-GPU Production Memory Calibration...")
    res = subprocess.run(["torchrun", "--nproc_per_node=2", "-m", "src.smoke_test", "--mode", "memory", "--data_root", valid_root, "--result_file", "test_mem.json", "--run_id", CURRENT_RUN_ID, "--config_hash", CURRENT_CONFIG_HASH], cwd=PROJECT_ROOT)
    with open(os.path.join(PROJECT_ROOT, "test_mem.json"), "r") as f:
        data = json.load(f)
        if data.get("status") == "PASS":
            mark_gate("MEMORY_CHECK", "PASS", f"Peak: {data.get('peak_allocated_gb')} GB")
        else:
            mark_gate("MEMORY_CHECK", "FAIL")
            raise RuntimeError(data.get("reason"))



## 16_checkpoint_test


In [21]:
if RUN_MODE in ["VALIDATE", "TRAIN"]:
    print("Running Checkpoint Write Test...")
    res = subprocess.run(["torchrun", "--nproc_per_node=2", "-m", "src.smoke_test", "--mode", "resume_a", "--data_root", valid_root, "--result_file", "test_ckpt.json", "--run_id", CURRENT_RUN_ID, "--config_hash", CURRENT_CONFIG_HASH], cwd=PROJECT_ROOT)
    with open(os.path.join(PROJECT_ROOT, "test_ckpt.json"), "r") as f:
        if json.load(f).get("status") == "PASS":
            mark_gate("CHECKPOINT_CHECK", "PASS")
        else:
            mark_gate("CHECKPOINT_CHECK", "FAIL")



## 17_resume_test


In [22]:
if RUN_MODE in ["VALIDATE", "TRAIN"]:
    print("Running Checkpoint Resume Test...")
    res = subprocess.run(["torchrun", "--nproc_per_node=2", "-m", "src.smoke_test", "--mode", "resume_b", "--data_root", valid_root, "--result_file", "test_res.json", "--run_id", CURRENT_RUN_ID, "--config_hash", CURRENT_CONFIG_HASH], cwd=PROJECT_ROOT)
    with open(os.path.join(PROJECT_ROOT, "test_res.json"), "r") as f:
        if json.load(f).get("status") == "PASS":
            mark_gate("RESUME_CHECK", "PASS")
        else:
            mark_gate("RESUME_CHECK", "FAIL")



## 18_final_gate


In [23]:
if RUN_MODE == "TRAIN":
    print("Running Preflight Dry Run (2-5 steps)...")

    # Build a Kaggle runtime config from the canonical experiment config.
    # The only environment-specific change is the actual dataset root
    # discovered by the notebook.
    with open(
        os.path.join(PROJECT_ROOT, "experiments/baseline_v1.json"), "r"
    ) as f:
        runtime_cfg = json.load(f)

    runtime_cfg["data"]["dataset_root"] = valid_root

    RUNTIME_CONFIG = os.path.join(
        PROJECT_ROOT,
        "experiments",
        "kaggle_runtime.json"
    )

    with open(RUNTIME_CONFIG, "w") as f:
        json.dump(runtime_cfg, f, indent=4)

    print("Runtime dataset root:")
    print(runtime_cfg["data"]["dataset_root"])
    print("Runtime config:")
    print(RUNTIME_CONFIG)

    res = subprocess.run([
        "torchrun",
        "--nproc_per_node=2",
        "-m",
        "src.train_ddp",
        "--config",
        RUNTIME_CONFIG,
        "--preflight",
        "--max_optimizer_steps",
        "5",
    ], cwd=PROJECT_ROOT, check=True)

    with open(
        os.path.join(PREFLIGHT_ROOT, "preflight_results.json"), "r"
    ) as f:
        pf_data = json.load(f)

    with open(
        os.path.join(PROJECT_ROOT, "experiments/baseline_v1.json"), "r"
    ) as f:
        canonical_cfg = json.load(f)

    for _m in list(sys.modules.keys()):
        if _m == "src" or _m.startswith("src."):
            del sys.modules[_m]
    from src.train_ddp import get_config_hash

    canonical_hash = get_config_hash(
        canonical_cfg,
        "model_config_hash"
    )

    if (
        pf_data.get("status") == "PASS"
        and pf_data.get("config_hash") == canonical_hash
    ):
        mark_gate(
            "PREFLIGHT_DRY_RUN_CHECK",
            "PASS",
            run_id=pf_data.get("run_id"),
            config_hash=pf_data.get("config_hash"),
        )
    else:
        mark_gate("PREFLIGHT_DRY_RUN_CHECK", "FAIL")
        raise RuntimeError(
            f"Preflight validation failed: {pf_data}"
        )

FINAL_STATUS = "PASS"
# Some gates are only for train/validate
if RUN_MODE in ["VALIDATE", "TRAIN"]:
    with open(os.path.join(PROJECT_ROOT, "experiments/baseline_v1.json"), "r") as f:
        canonical_cfg = json.load(f)
    for _m in list(sys.modules.keys()):
        if _m == "src" or _m.startswith("src."):
            del sys.modules[_m]
    from src.train_ddp import get_config_hash
    CURRENT_CONFIG_HASH = get_config_hash(canonical_cfg, "model_config_hash")
    CURRENT_RUN_ID = canonical_cfg.get("run_id") or "test_run_id"

    # 9. Current-run evidence validation
    import time
    required_files = [
        "test_2gpu.json", "test_mem.json", "test_ckpt.json", "test_res.json"
    ]
    for filename in required_files:
        fpath = os.path.join(PROJECT_ROOT, filename)
        if not os.path.exists(fpath):
            print(f"Missing required test result: {filename}")
            FINAL_STATUS = "FAIL"
            continue
        with open(fpath, "r") as f:
            data = json.load(f)
            if data.get("status") != "PASS":
                print(f"Test result not PASS: {filename}")
                FINAL_STATUS = "FAIL"
            if data.get("run_id") != CURRENT_RUN_ID or data.get("config_hash") != CURRENT_CONFIG_HASH:
                print(f"Evidence mismatch (run_id/config_hash) in {filename}")
                FINAL_STATUS = "FAIL"
        
        # Freshness check: file must be modified recently (within the last hour)
        mtime = os.path.getmtime(fpath)
        if time.time() - mtime > 3600:
            print(f"Stale test result (timestamp): {filename}")
            FINAL_STATUS = "FAIL"
            
    if RUN_MODE == "TRAIN":
        fpath = os.path.join(PREFLIGHT_ROOT, "preflight_results.json")
        if os.path.exists(fpath):
            with open(fpath, "r") as f:
                pf_data = json.load(f)
                # NOTE: preflight_results.json's run_id is generated fresh per-run
                # by src.experiment.setup_experiment_run() (real timestamp+uuid) --
                # it is NOT the CURRENT_RUN_ID placeholder ("test_run_id") used to
                # tag the isolated smoke-test artifacts above, which is only ever
                # set from canonical_cfg["run_id"] (always blank in baseline_v1.json).
                # Comparing against CURRENT_RUN_ID here would fail unconditionally,
                # regardless of correctness. What actually matters: the preflight
                # really ran (a run_id was assigned) and its config hash matches
                # the canonical config -- both of which are meaningful checks.
                if not pf_data.get("run_id") or pf_data.get("config_hash") != CURRENT_CONFIG_HASH:
                    print(f"Evidence mismatch in preflight_results.json")
                    FINAL_STATUS = "FAIL"
            mtime = os.path.getmtime(fpath)
            if time.time() - mtime > 3600:
                print(f"Stale preflight result (timestamp).")
                FINAL_STATUS = "FAIL"

    MODE_REQUIRED_GATES = {
        "VALIDATE": [
            "ENV_CHECK", "GPU_CHECK", "DATA_CHECK", "PROJECT_CHECK", "DEPENDENCY_CHECK",
            "PVT_CHECK", "STATIC_CHECK", "TRANSFORM_CHECK", "MODEL_CHECK", "MOE_CHECK",
            "LOSS_CHECK", "OPT_CHECK", "DDP_CHECK", "MEMORY_CHECK",
            "CHECKPOINT_CHECK", "RESUME_CHECK"
        ],
        "TRAIN": [
            "ENV_CHECK", "GPU_CHECK", "DATA_CHECK", "PROJECT_CHECK", "DEPENDENCY_CHECK",
            "PVT_CHECK", "STATIC_CHECK", "TRANSFORM_CHECK", "MODEL_CHECK", "MOE_CHECK",
            "LOSS_CHECK", "OPT_CHECK", "DDP_CHECK", "MEMORY_CHECK",
            "CHECKPOINT_CHECK", "RESUME_CHECK", "PREFLIGHT_DRY_RUN_CHECK"
        ],
        "RESUME": [
            "ENV_CHECK", "GPU_CHECK", "DATA_CHECK", "PROJECT_CHECK", "DEPENDENCY_CHECK",
            "STATIC_CHECK", "RESUME_PREFLIGHT_CHECK"
        ]
    }
    
    required = MODE_REQUIRED_GATES.get(RUN_MODE, [])
    for k in required:
        v = GATES.get(k)
        if v != "PASS":
            print(f"GATE {k} FAILED or NOT RUN.")
            FINAL_STATUS = "FAIL"

    with open(os.path.join(PROJECT_ROOT, "final_audit.json"), "w") as f:
        json.dump({
            "timestamp": time.time(),
            "gates": GATES,
            "final_status": FINAL_STATUS
        }, f, indent=4)

print(f"FINAL AUDIT STATUS: {FINAL_STATUS}")



FINAL AUDIT STATUS: PASS


In [24]:
# ls -lah /kaggle/working/WXSOD_Preflight/

In [25]:
import os

print("Preflight files:")
for root, dirs, files in os.walk("/kaggle/working/WXSOD_Preflight"):
    for f in files:
        print(os.path.join(root, f))

print("\nAll preflight result files:")
os.system('find /kaggle/working -name "preflight_results.json" -o -name "final_config.json"')

Preflight files:
/kaggle/working/WXSOD_Preflight/final_config.json
/kaggle/working/WXSOD_Preflight/final_config_sha256
/kaggle/working/WXSOD_Preflight/preflight_results.json

All preflight result files:
/kaggle/working/WXSOD_Checkpoints/final_config.json
/kaggle/working/WXSOD_Preflight/final_config.json
/kaggle/working/WXSOD_Preflight/preflight_results.json


0

## 19_train


In [26]:
if RUN_MODE == "TRAIN":
    if FINAL_STATUS != "PASS" or GATES.get("PREFLIGHT_DRY_RUN_CHECK") != "PASS":
        raise RuntimeError("Refusing to train: Not all gates passed.")

    with open(
        os.path.join(PROJECT_ROOT, "experiments/baseline_v1.json"), "r"
    ) as f:
        canonical_cfg = json.load(f)

    for _m in list(sys.modules.keys()):
        if _m == "src" or _m.startswith("src."):
            del sys.modules[_m]
    from src.train_ddp import get_config_hash

    canonical_hash = get_config_hash(
        canonical_cfg,
        "model_config_hash"
    )

    with open(
        os.path.join(PREFLIGHT_ROOT, "preflight_results.json"), "r"
    ) as f:
        pf_data = json.load(f)

    actual_hash = pf_data.get("config_hash")

    print(f"Validated config hash: {canonical_hash}")
    print(f"Training config hash:  {actual_hash}")

    if canonical_hash != actual_hash:
        raise RuntimeError(
            "Config hash mismatch between canonical config and preflight runtime!"
        )

    print("MATCH")
    print("Launching final 50-epoch training...")

    subprocess.run([
        "torchrun",
        "--nproc_per_node=2",
        "-m",
        "src.train_ddp",
        "--config",
        RUNTIME_CONFIG,
    ], cwd=PROJECT_ROOT, check=True)

## 20_status


In [27]:
if RUN_MODE == "TRAIN":
    if os.path.exists(os.path.join(CHECKPOINT_ROOT, "training_complete.json")):
        print("Training successfully reached completion state.")
    else:
        print("Training did not produce completion marker.")



## 21_resume


In [ ]:
if RUN_MODE == "RESUME":
    print("Initiating Resume Recovery Sequence...")
    local_latest = os.path.join(CHECKPOINT_ROOT, "latest.pth")

    if os.path.exists(os.path.join(CHECKPOINT_ROOT, "training_complete.json")):
        raise RuntimeError("TRAINING ALREADY COMPLETE. Cannot resume.")

    if not os.path.exists(local_latest):
        # Cell 1 already tried to fetch this from HF at notebook start; if it's
        # still missing here there's genuinely no checkpoint to resume from.
        raise RuntimeError(
            f"{local_latest} not found. No checkpoint is available on "
            f"{HF_REPO_ID} either (see the CHECKPOINT_CHECK gate above). "
            "Start a fresh TRAIN run instead of RESUME."
        )

    with open(os.path.join(PROJECT_ROOT, "experiments/baseline_v1.json"), "r") as f:
        runtime_cfg = json.load(f)

    runtime_cfg["data"]["dataset_root"] = valid_root
    RUNTIME_CONFIG = os.path.join(PROJECT_ROOT, "experiments", "kaggle_runtime.json")

    with open(RUNTIME_CONFIG, "w") as f:
        json.dump(runtime_cfg, f, indent=4)

    print("Running Resume Preflight Dry Run...")
    subprocess.run([
        "torchrun", "--nproc_per_node=2", "-m", "src.train_ddp",
        "--config", RUNTIME_CONFIG,
        "--resume", "latest", "--preflight", "--max_optimizer_steps", "5"
    ], cwd=PROJECT_ROOT, check=True)
    with open(os.path.join(PREFLIGHT_ROOT, "preflight_results.json"), "r") as f:
        pf_data = json.load(f)
    if pf_data.get("status") == "PASS":
        mark_gate("RESUME_PREFLIGHT_CHECK", "PASS", run_id=pf_data.get("run_id"), config_hash=pf_data.get("config_hash"))
    else:
        mark_gate("RESUME_PREFLIGHT_CHECK", "FAIL")
        raise RuntimeError("Resume preflight failed.")

    print("Executing Torchrun Resume...")
    # train_ddp.py reads HF_REPO_ID / HF_TOKEN / CHECKPOINT_ROOT from the
    # environment (set in Cell 1) and pushes latest.pth / best.pth to HF
    # in the background as they're written — no separate push step needed.
    subprocess.run([
        "torchrun", "--nproc_per_node=2", "-m", "src.train_ddp",
        "--config", RUNTIME_CONFIG,
        "--resume", "latest"
    ], cwd=PROJECT_ROOT, check=True)


## 22_evaluate


In [ ]:
import sys

if RUN_MODE in ["TRAIN", "EVALUATE"]:
    print("Evaluating Best Checkpoint...")
    best_ckpt = "/kaggle/working/WXSOD_Checkpoints/best.pth"
    if not os.path.exists(best_ckpt):
        best_ckpt = os.path.join(CHECKPOINT_ROOT, "latest.pth")
    data_dir = "/kaggle/working/WXSDO_data/WXSDO_data"
    eval_out_dir = "/kaggle/working/WXSOD_EvalResults"
    os.makedirs(eval_out_dir, exist_ok=True)
    print("Started the Process")

    process = subprocess.Popen(
        [
            "python", "-u", "-m", "src.evaluate",
            "--checkpoint", best_ckpt,
            "--dataset", "both",
            "--data_dir", data_dir,
            "--out_dir", eval_out_dir,
        ],
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )

    while True:
        chunk = process.stdout.read(1)
        if not chunk and process.poll() is not None:
            break
        if chunk:
            sys.stdout.write(chunk.decode(errors="replace"))
            sys.stdout.flush()

    retcode = process.wait()
    if retcode != 0:
        raise subprocess.CalledProcessError(retcode, process.args)

    print("\n--- Files in eval_out_dir ---")
    for f in os.listdir(eval_out_dir):
        print(f)

Evaluating Best Checkpoint...
Started the Process
--- EVALUATION CONFIGURATION ---
Checkpoint: /kaggle/working/WXSOD_Checkpoints/best.pth
Epoch: 14
Best Metric: 0.01955700641716319
Model Hash: 2cd252ad3a3581e1c65961b828857d70
TTA: none
--------------------------------


--- Weather Distribution ---
TRAIN: {'rainafog': 1265, 'snow': 1232, 'fog': 1228, 'rain': 1210, 'snowafog': 1255, 'dark': 1230, 'light': 1217, 'rainasnow': 1197, 'clean': 508}
VALIDATION: {'light': 314, 'rain': 314, 'clean': 123, 'rainafog': 297, 'fog': 306, 'snowafog': 278, 'rainasnow': 297, 'snow': 315, 'dark': 305}
----------------------------

Evaluating test_sys...
/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 instead!")
Evaluating:   0%|          | 0/1500 [00:00<?, ?it/s]valuating:   0%|          | 1/1500 [00:01<29:59,  1.

In [ ]:
import torch

# Path to your best checkpoint
checkpoint_path = "/kaggle/working/WXSOD_Checkpoints/best.pth"

try:
    # Added weights_only=False here
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    
    print(f"✅ Current Best Model is from Epoch: {checkpoint['epoch'] + 1}")
    print(f"🏆 Current Best MAE: {checkpoint['best_metric']:.4f}")
except Exception as e:
    print(f"Error loading checkpoint: {e}")


In [ ]:
# Push evaluation results to Hugging Face. Token comes from the Kaggle
# Secret set up in Cell 1 — never paste a token here.
import subprocess, os

eval_dir = "/kaggle/working/spatial_moe_sod/evaluation/best"
zip_out = "/kaggle/working/best.zip"

subprocess.run(["rm", "-f", zip_out], check=False)
subprocess.run(["zip", "-0", "-r", zip_out, "best"], cwd=os.path.dirname(eval_dir), check=True)
subprocess.run(["unzip", "-l", zip_out], check=False)

from huggingface_hub import HfApi
HfApi(token=HF_TOKEN).upload_file(
    path_or_fileobj=zip_out,
    path_in_repo="results/best.zip",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
print(f"Uploaded {zip_out} -> {HF_REPO_ID}/results/best.zip")
